### 1. Initializing Spark Session and Loading Raw Data Locally
This cell imports the necessary components from PySpark and the Python `os` module to dynamically locate the local `employees.txt` file next to this notebook. It initializes a `SparkSession` and reads the file as a DataFrame, which is then converted into an RDD so we can safely filter out the header row using a lambda function.

In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

spark = SparkSession.builder.appName("EmployeeAnalysis").getOrCreate()

current_dir = os.getcwd()
local_file_path = f"file://{current_dir}/employees.txt"

raw_data = spark.read.text(local_file_path)
header = raw_data.first()[0]

data_rdd = raw_data.rdd.filter(lambda line: line[0] != header)

### 2. Task 1: Parse Text Data into a Structured Format
In this cell, we define a helper function `parse_line` to split each comma-separated row into individual fields while safely handling corrupted rows. We map this function across our RDD, filter out any `None` values, and convert the structured RDD into a clean PySpark DataFrame with proper schema names.

In [14]:
# Enhanced helper function to parse and convert data types safely
def parse_line(line):
    parts = line[0].split(',')
    
    # Check if the line has the baseline layout structure
    if len(parts) >= 6:
        try:
            emp_id = int(parts[0])
            name = parts[1]
            department = parts[2]
            job_title = parts[3]
            salary = float(parts[4])
            location = parts[5]
            
            return (emp_id, name, department, job_title, salary, location)
        except ValueError:
            # Safely catch data type mismatch errors (like text in salary column)
            return None
    return None

# Map the parsing function and exclude corrupted or invalid records
structured_rdd = data_rdd.map(parse_line).filter(lambda x: x is not None)

# Convert the structured RDD into a PySpark DataFrame with defined columns
df = structured_rdd.toDF(["emp_id", "name", "department", "job_title", "salary", "location"])

In [16]:
df.show()

+------+----------------+-----------+-----------------+--------+-------------+
|emp_id|            name| department|        job_title|  salary|     location|
+------+----------------+-----------+-----------------+--------+-------------+
|     1|      John Smith|Engineering| Senior Developer|125000.0|San Francisco|
|     2|   Sarah Johnson|      Sales|Account Executive| 85000.0|     New York|
|     3|Michael Williams|Engineering|Software Engineer| 95000.0|       Austin|
|     4|  Jennifer Brown|  Marketing|Marketing Manager| 92000.0|      Chicago|
|     5|     David Jones|    Finance|   Senior Analyst|105000.0|       Boston|
|     6|     Lisa Garcia|         IT|  DevOps Engineer|115000.0|      Seattle|
|     8| Patricia Wilson|         HR|       HR Manager| 88000.0|       Denver|
|     9|  James Anderson|      Sales|    Sales Manager|110000.0|      Atlanta|
|    10|     Mary Thomas|Engineering|        Tech Lead|145000.0|      Seattle|
+------+----------------+-----------+---------------

### 3. Task 2 & Task 3: Name Occurrences and Safe Record Filtering
This step aggregates the dataset to count how many times each employee name appears using the `groupBy` transformation. Additionally, it applies a secondary defensive data-quality check by filtering out any records where the salary column contains negative numbers or zero values.

In [18]:
name_counts = df.groupBy("name").count()
valid_df = df.filter(col("salary") > 0)

In [19]:
name_counts.show()

+----------------+-----+
|            name|count|
+----------------+-----+
|  Jennifer Brown|    1|
|  James Anderson|    1|
|Michael Williams|    1|
|   Sarah Johnson|    1|
| Patricia Wilson|    1|
|      John Smith|    1|
|     David Jones|    1|
|     Lisa Garcia|    1|
|     Mary Thomas|    1|
+----------------+-----+



In [20]:
valid_df.show()

+------+----------------+-----------+-----------------+--------+-------------+
|emp_id|            name| department|        job_title|  salary|     location|
+------+----------------+-----------+-----------------+--------+-------------+
|     1|      John Smith|Engineering| Senior Developer|125000.0|San Francisco|
|     2|   Sarah Johnson|      Sales|Account Executive| 85000.0|     New York|
|     3|Michael Williams|Engineering|Software Engineer| 95000.0|       Austin|
|     4|  Jennifer Brown|  Marketing|Marketing Manager| 92000.0|      Chicago|
|     5|     David Jones|    Finance|   Senior Analyst|105000.0|       Boston|
|     6|     Lisa Garcia|         IT|  DevOps Engineer|115000.0|      Seattle|
|     8| Patricia Wilson|         HR|       HR Manager| 88000.0|       Denver|
|     9|  James Anderson|      Sales|    Sales Manager|110000.0|      Atlanta|
|    10|     Mary Thomas|Engineering|        Tech Lead|145000.0|      Seattle|
+------+----------------+-----------+---------------

### 4. Task 4 & Task 5: Calculate Average Salary and Employee Count per Department
This cell performs key business metrics calculations grouped by organizational departments. It computes the mathematical average salary for each department and determines the total headcount by counting unique employee IDs within those same groups.

In [21]:
avg_salary = valid_df.groupBy("department").agg(avg("salary").alias("average_salary"))
emp_count_per_dept = valid_df.groupBy("department").agg(count("emp_id").alias("employee_count"))

### 5. Displaying Aggregated Metrics
This final cell triggers the execution of our transformation pipelines using the `.show()` action. It outputs structured, human-readable preview tables containing the calculated department salary averages and employee distribution metrics directly below the cell.

In [24]:
print("Average Salary per Department:")
avg_salary.show()

Average Salary per Department:
+-----------+------------------+
| department|    average_salary|
+-----------+------------------+
|      Sales|           97500.0|
|Engineering|121666.66666666667|
|         HR|           88000.0|
|    Finance|          105000.0|
|  Marketing|           92000.0|
|         IT|          115000.0|
+-----------+------------------+



In [ ]:
print("Employee Count per Department:")
emp_count_per_dept.show()